In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import shutil

INPUT_ROOT = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

bench_source = None
verifier_source = None

for root, directories, files in os.walk(INPUT_ROOT):
    for filename in files:
        full_path = os.path.join(root, filename)

        if filename == "bench_report.json":
            bench_source = full_path

        if filename.endswith(".py"):
            try:
                with open(
                    full_path,
                    "r",
                    encoding="utf-8"
                ) as file:
                    content = file.read()

                if "cost_report.json" in content:
                    verifier_source = full_path
            except Exception:
                pass

assert bench_source is not None, "bench_report.json was not found"
assert verifier_source is not None, "Cost verifier was not found"

shutil.copy(
    bench_source,
    os.path.join(WORKING_DIR, "bench_report.json")
)

shutil.copy(
    verifier_source,
    os.path.join(WORKING_DIR, "verify.py")
)

os.chdir(WORKING_DIR)

print("bench_report.json FOUND")
print("verify.py FOUND")
print("EXTRA LAB FILES READY")

bench_report.json FOUND
verify.py FOUND
EXTRA LAB FILES READY


In [2]:
import json
import math

BENCH_PATH = "/kaggle/working/bench_report.json"
COST_REPORT_PATH = "/kaggle/working/cost_report.json"

GPU_HOURLY_USD = 0.35
TARGET_P95_S = 8.0

with open(BENCH_PATH, "r", encoding="utf-8") as file:
    benchmark = json.load(file)

source_levels = benchmark["runs"][-1]["levels"]


def cost_per_million_tokens(
    tokens_per_s,
    gpu_hourly_usd
):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = (
        tokens_per_hour / 1_000_000
    )

    return round(
        gpu_hourly_usd / million_tokens_per_hour,
        4
    )


levels = []

for source_level in source_levels:
    level = dict(source_level)

    level["cost_per_million_tokens_usd"] = (
        cost_per_million_tokens(
            level["tokens_per_s"],
            GPU_HOURLY_USD
        )
    )

    levels.append(level)


under_target = [
    level
    for level in levels
    if level["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under_target,
    key=lambda level: level["concurrency"]
)

scale_out_plan = []

for multiple in (1.0, 1.5, 2.0, 3.0):
    required_tokens_per_s = round(
        knee["tokens_per_s"] * multiple,
        6
    )

    replicas = math.ceil(
        required_tokens_per_s
        / knee["tokens_per_s"]
        - 1e-9
    )

    scale_out_plan.append(
        {
            "demand_multiple": multiple,
            "required_tokens_per_s": required_tokens_per_s,
            "replicas_needed": replicas,
            "total_hourly_cost_usd": round(
                replicas * GPU_HOURLY_USD,
                2
            ),
            "effective_p95_s": knee["latency_p95_s"]
        }
    )


cost_report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_out_plan
}

with open(
    COST_REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(cost_report, file, indent=2)

print("KNEE:", knee["concurrency"])
print(
    "COST AT KNEE:",
    knee["cost_per_million_tokens_usd"],
    "USD per million tokens"
)

print("\nSCALE-OUT PLAN:")

for row in scale_out_plan:
    print(row)

print("\nCOST REPORT CREATED")

KNEE: 32
COST AT KNEE: 0.1333 USD per million tokens

SCALE-OUT PLAN:
{'demand_multiple': 1.0, 'required_tokens_per_s': 729.52, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.8438}
{'demand_multiple': 1.5, 'required_tokens_per_s': 1094.28, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.8438}
{'demand_multiple': 2.0, 'required_tokens_per_s': 1459.04, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.8438}
{'demand_multiple': 3.0, 'required_tokens_per_s': 2188.56, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.8438}

COST REPORT CREATED


In [3]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "/kaggle/working/verify.py"
    ],
    cwd="/kaggle/working",
    text=True
)

print("VERIFIER EXIT CODE:", result.returncode)

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
VERIFIER EXIT CODE: 0


In [4]:
import base64
from IPython.display import HTML, display

file_path = "/kaggle/working/cost_report.json"

with open(file_path, "rb") as file:
    encoded = base64.b64encode(file.read()).decode()

download_link = f"""
<a download="cost_report.json"
   href="data:application/json;base64,{encoded}">
   Download cost_report.json
</a>
"""

display(HTML(download_link))